In [ ]:
# ===== Pacotes / Packages =====
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedGroupKFold, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, ConfusionMatrixDisplay

In [ ]:
# ===== 1. Ler os dados / Read the data .XLSX =====
dados = pd.read_excel("NAME.xlsx")
print(dados.head())

In [ ]:
# ===== 1. Ler os dados / Read the data .CSV =====
dados = pd.read_csv('NAME.csv', sep=';')  # especifica o separador / specifies the separator
print(dados.head())

Se amostras independentes (não agrupadas) / For independent (ungrouped) samples.

In [ ]:
# A classe deve estar na primeira coluna / The class must be in the first column
dados["Classe"] = dados.iloc[:, 0].astype(str)  # converter para string/categoria / convert to string/category
dados = dados.drop(dados.columns[0], axis=1)    # remover a coluna original / remove the original column
cols = list(dados.columns)
cols = cols[:-1] + ["Classe"]                   # garantir que "Classe" fique por último / ensure that "Class" comes last
dados = dados[cols]

# ===== 2. Garantir que preditoras sejam numéricas / Ensure predictors are numeric =====
dados_pred = dados.drop("Classe", axis=1).apply(pd.to_numeric, errors="coerce")

# Remover colunas constantes / Remove constant columns
constantes = [c for c in dados_pred.columns if dados_pred[c].nunique() == 1]
dados_pred = dados_pred.drop(columns=constantes)

# Remover colunas com NA / Remove columns with NA
dados_pred = dados_pred.dropna(axis=1)

# Recombinar com a classe / Recombine with the class
dados = pd.concat([dados_pred, dados["Classe"]], axis=1)

In [ ]:
# ===== 3. Divisão treino/teste / Training/testing division =====
X = dados.drop("Classe", axis=1).values
y = dados["Classe"].values

# Codificar classes como números / Encode classes as numbers
le = LabelEncoder()
y = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=1234, stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

In [ ]:
# ===== Validação Cruzada no Treino / Cross-Validation on Training Set =====

# Instanciar o modelo QDA para a validação cruzada
modelo_cv = QuadraticDiscriminantAnalysis()

# Definir o esquema de validação cruzada
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=1234)

# Executar a validação cruzada usando acurácia como métrica
scores_cv = cross_val_score(modelo_cv, X_train, y_train, cv=cv, scoring='accuracy')

print("Acurácias por fold:", scores_cv)
print(f"Acurácia média da Validação Cruzada: {scores_cv.mean():.4f} (+/- {scores_cv.std():.4f})")

Se amostras dependentes (agrupadas) / For dependent (grouped) samples.

In [ ]:
# ===== Ajustar Colunas: Grupo, Classe e Preditoras =====

# Vamos renomear e organizar para clareza
grupos = dados.iloc[:, 0].astype(str).values  # ID dos grupos (Well)
classes = dados.iloc[:, 1].astype(str).values  # Classe (Growth)

# Manter um registro das variáveis de identificação
dados["Grupo"] = grupos
dados["Classe"] = classes

# ===== 2. Garantir que preditoras sejam numéricas =====
# As preditoras vão da 3ª coluna em diante
features_cols = dados.columns[2:-2] # desconsiderando as colunas de ID originais e as recém-criadas 'Grupo' e 'Classe'
dados_pred = dados[features_cols].apply(pd.to_numeric, errors="coerce")

# Remover colunas constantes
constantes = [c for c in dados_pred.columns if dados_pred[c].nunique() == 1]
dados_pred = dados_pred.drop(columns=constantes)

# Remover colunas com NA
dados_pred = dados_pred.dropna(axis=1)

# Recombinar preditoras com Grupo e Classe
dados_final = pd.concat([dados_pred, dados[["Grupo", "Classe"]]], axis=1)

In [ ]:
# ===== 3. Divisão Treino/Teste usando StratifiedGroupKFold =====

X = dados_final.drop(columns=["Grupo", "Classe"]).values
y = dados_final["Classe"].values
groups = dados_final["Grupo"].values

# Codificar classes como números
le = LabelEncoder()
y = le.fit_transform(y)

# Para fazer uma divisão limpa de treino (80%) e teste (20%) respeitando grupos,
# usaremos o StratifiedGroupKFold com 5 splits e pegaremos o primeiro split para teste.
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=1234)
train_idx, test_idx = next(sgkf.split(X, y, groups))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
groups_train, groups_test = groups[train_idx], groups[test_idx]

print("Train shapes:", X_train.shape, y_train.shape)
print("Test shapes:", X_test.shape, y_test.shape)
print("Número de grupos únicos no Treino:", len(np.unique(groups_train)))
print("Número de grupos únicos no Teste:", len(np.unique(groups_test)))
# Garantir que não há intersecção de grupos entre treino e teste
assert len(set(groups_train).intersection(set(groups_test))) == 0, "Erro: Há grupos compartilhados entre treino e teste!"

In [ ]:
# ===== Validação Cruzada no Treino usando StratifiedGroupKFold =====

# Instanciar o modelo QDA para a validação cruzada
modelo_cv = QuadraticDiscriminantAnalysis()

# Definir o esquema de validação cruzada respeitando os grupos do treino
cv_group = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=1234)

# Executar a validação cruzada passando os grupos de treino explicitamente
scores_cv = cross_val_score(
    modelo_cv,
    X_train,
    y_train,
    groups=groups_train,
    cv=cv_group,
    scoring='accuracy'
)

print("Acurácias por fold na Validação Cruzada:", scores_cv)
print(f"Acurácia média da Validação Cruzada (Agrupada): {scores_cv.mean():.4f} (+/- {scores_cv.std():.4f})")

Continue

In [ ]:
# ===== 4. Rodar QDA / Run QDA =====
modelo_qda = QuadraticDiscriminantAnalysis()
modelo_qda.fit(X_train, y_train)

# ===== 5. Predição / Prediction =====
pred_treino = modelo_qda.predict(X_train)
pred_teste = modelo_qda.predict(X_test)

print("Training Accuracy:", accuracy_score(y_train, pred_treino))
print("Test Accuracy:", accuracy_score(y_test, pred_teste))

In [ ]:
# ===== 6. Matriz de Confusão / Confusion Matrix =====
y_test_int = y_test.astype(int)
pred_teste_int = pred_teste.astype(int)

cm = confusion_matrix(y_test_int, pred_teste_int, labels=np.unique(y_test_int))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
disp.plot(cmap="Blues", values_format="d")
plt.title("Confusion Matrix -Test Set")
plt.show()

# Relatório detalhado / Detailed report (precision, recall, f1-score)
print("\nClassification Report:")
print(classification_report(y_test, pred_teste, target_names=le.classes_))
